<img src="logo.png" alt="Vegeta" width="240">

# AGUYA — a turbojet sampler that reaches the fire at 150 m/s (29)

MERLIN (notebooks 26–27) reaches a fire 10 km away in about 2.6 minutes at best, whatever the propulsor, because a
battery cannot feed more than a few kilowatts and a propeller runs out of tip speed. **AGUYA** does MERLIN's job (dash to a
reported hotspot, crosswind passes through the smoke at 150 m with a forward gas sensor, come home) with the model
turbojet of notebook 28, at about 150–165 m/s.

What changes when the energy store is kerosene:
- **it burns off**: the aircraft is lightest on the way home;
- **it is sized per mission**: the tank, and so the fuselage, grows with the range;
- **the engine is hot, and so is its jet**: the tail must stay out of the jet, and the sensor out of the exhaust;
- **the speed changes the structure**: at 150 m/s a 7.5 m/s gust is a 10–14 g event for a light wing, and the dynamic
  pressure is 14 kPa;
- **recovery is by parachute**: no belly landing with a hot engine and fuel over dry forest.

```
1  mission and mass budget (edit here)            6  the mission flown: speed, height, mass, fuel against time
2  AGUYA in CAD (Dedalus): pod, pylon, V-tail      7  the hot jet: does it reach the V-tail? (estimate, then CFD)
3  drag, thrust, speeds, the catapult              8  does AGUYA sample its own exhaust?
4  fuel and tank for 5–30 km                       9  the wing at the dash: gust loads (FEA) and vibration modes
5  the race against MERLIN                        10  export → designs/data/aguya_design.json
```

The engine comes from notebook 28's export (`designs/data/microjet.json`), and MERLIN's numbers from notebooks 25–27's
libraries. If either is missing, the engine is recalibrated here from its catalogue class and the MERLIN comparison is
skipped. `VEGETA_SKIP_OPENFOAM=1` skips the CFD.

In [ ]:
import json, math, os, shutil, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from vegeta import dedalus, talos, aeromant, core
from vegeta.boreas import microjet as mj
from vegeta.dedalus import viz as dviz
from vegeta.talos import viz as tviz
sys.path.insert(0, "designs")
import aguya as ag
import aguya_flight as F

ROOT = Path("_runs/aguya"); shutil.rmtree(ROOT, ignore_errors=True); ROOT.mkdir(parents=True)
DATA = Path("designs/data")
RUN_CFD = os.environ.get("VEGETA_SKIP_OPENFOAM") != "1"
QUICK = os.environ.get("AGUYA_QUICK") == "1"
CPU = max(1, min(8, os.cpu_count() or 1))
RHO, G, A0 = 1.225, 9.80665, 340.3
JET_PATH = DATA / "microjet.json"
if JET_PATH.is_file():
    JET = mj.load(JET_PATH)
    ENGINE, UNIT = JET["engine"], F.jet_unit_from_export(JET)
    print(f"engine from notebook 28: {JET['engine_class']} ({JET_PATH})")
else:
    ENGINE = mj.from_catalogue("140 N class"); UNIT = F.jet_unit(ENGINE); JET = None
    print("no designs/data/microjet.json (run notebook 28): the 140 N class engine, calibrated here")
AGUYA_COLOR, MERLIN_COLOR = "#2ca02c", {"edf": "#d62728", "tractor": "#1f77b4", "pusher": "#ff7f0e"}

## 1. Mission and mass budget — edit here

Same job as MERLIN: sampling at 150 m above the launch site, 3 minutes of passes, 15 % of the fuel kept in reserve.

The dry mass replaces MERLIN's printed shells with a fuel-proof composite airframe. The 14 kPa dynamic pressure and
the hot engine bay rule out LW-PLA. The dry mass adds the engine and its ECU, pump and valves, a 2S pack for the ECU,
servos and avionics, and a parachute. The engine starts and warms up on the catapult (45 s at idle) and is shut down
when the parachute opens.

In [ ]:
MISSION = F.JetMission(altitude_m=150.0, sampling_s=180.0, sampling_bank_deg=30.0, start_s=45.0, recovery_s=40.0,
                       launch_margin=1.25, reserve=0.15, dash_speed=None)     # dash_speed None: 98 % of the top speed
DISTANCES_KM = (5, 8, 10, 20, 30)
DESIGN_RANGE_KM = 30                    # the tank (and the fuselage) is sized for this one
BUDGET = dict(F.DRY_MASS_KG)
display(pd.DataFrame({"mass [g]": {**{k: v * 1000 for k, v in BUDGET.items()},
                                   f"engine + ECU, pump, valves ({ENGINE.name})": UNIT.mass_kg * 1000}}))
print(f"dry mass {F.dry_mass(UNIT, BUDGET):.2f} kg; static thrust {UNIT.full(0)[0]:.0f} N")

## 2. AGUYA in CAD (Dedalus)

`designs/aguya.py` is notebook 09's `FixedWing`: its wing sections and centre section. Everything else is AGUYA's own:
- an ogive sensor nose, a fuselage whose middle is the fuel tank, and a tail cone;
- the engine pod on a pylon behind the wing, 14 mm wider than the casing, with its intake at the engine's bell-mouth
  and the nozzle exit from the cycle's `A8`;
- a V-tail whose two halves stand clear of the jet.

`ag.for_engine` sizes the pod around notebook 28's engine. The fuselage diameter comes from section 4 (the tank). This
cell draws the starting point.

In [ ]:
design_file = ROOT / "aguya.py"
shutil.copy(Path("designs/aguya.py"), design_file)
aguya = dedalus.load_design(f"{design_file}:Aguya")
P0 = ag.for_engine(ENGINE)
display(pd.DataFrame(aguya.params.table()).set_index("name"))
GEOM = aguya.generate(**P0)
print(f"valid {GEOM.shape.isValid()}, {len(GEOM.shape.Solids())} solid, {np.round(GEOM.dimensions, 0)} mm;"
      f" tank {ag.tank_volume_l(P0):.2f} L usable")
try:
    dviz.show(dviz.plot3d(GEOM))
except ImportError as e:
    print("3D view needs pyvista:", e)

def side_view(p, ax, title):
    o = ag.outline(p)
    for poly, c in zip(o["side"], ("#9fb4c7", "#7f7f7f", "#9fb4c7", "#9fb4c7")):
        ax.fill(poly[:, 0], poly[:, 1], color=c, alpha=0.6, ec="k", lw=0.6)
    ax.plot([o["nozzle_x"], o["nozzle_x"] + 0.6], [o["nozzle_z"]] * 2, color="#d62728", lw=6, alpha=0.35, label="jet axis")
    ax.set_aspect("equal"); ax.grid(alpha=0.3); ax.set_title(title, fontsize=10); ax.legend(loc="lower left", fontsize=8)

fig, ax = plt.subplots(figsize=(13, 3.2)); side_view(P0, ax, "AGUYA side view (start point)"); ax.set_xlabel("x [m]");

## 3. Drag, thrust, speeds and the catapult

The parasite drag is the component build-up (`ag.drag_buildup`, Raymer's friction and form factors with his
compressibility corrections at Mach 0.44), plus an allowance for the intake spillage, the hatch and the antennas.
Full-throttle thrust against drag gives the top speed. The stall speed with full fuel sets the catapult.

In [ ]:
AF0 = F.airframe(P0, UNIT, budget=BUDGET)
m_full, m_dry = AF0.dry_mass_kg + AF0.tank_kg, AF0.dry_mass_kg
b = ag.drag_buildup(P0, 150.0, mach=150.0 / A0)
display(pd.DataFrame({"Cd·A [cm²]": {k: v * 1e4 for k, v in b["parts_m2"].items()}}).round(2))
V = np.linspace(10, 210, 81)
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for frac, ls in ((1.0, "-"), (0.8, "--"), (0.6, ":")):
    T = [np.interp(frac * (UNIT.rpm[-1] - UNIT.rpm[0]) + UNIT.rpm[0], UNIT.rpm, UNIT._row(v)[0]) for v in V]
    ax[0].plot(V, T, color=AGUYA_COLOR, ls=ls, label=f"thrust, {frac:.0%} of the speed range")
for m, c in ((m_full, "k"), (m_dry, "0.5")):
    ax[0].plot(V, [AF0.drag(v, m) for v in V], color=c, label=f"drag at {m:.1f} kg")
ax[0].set(xlabel="airspeed [m/s]", ylabel="N", title="thrust and drag", ylim=(0, 1.3 * UNIT.full(0)[0])); ax[0].legend(fontsize=8)
ax[1].plot(V, [UNIT.for_thrust(v, AF0.drag(v, m_full))[0] * 60e3 for v in V], color=AGUYA_COLOR)
ax[1].set(xlabel="airspeed [m/s]", ylabel="fuel flow [g/min]", title="fuel flow in level flight (full tank)")
for a in ax: a.grid(alpha=0.3)
fig.tight_layout()
SPEEDS = {"top speed, full tank [m/s]": F.top_speed(AF0, UNIT, m_full), "top speed, dry [m/s]": F.top_speed(AF0, UNIT, m_dry),
          "stall, full tank [m/s]": AF0.stall_speed(m_full), "catapult release [m/s]": MISSION.launch_margin * AF0.stall_speed(m_full),
          "dynamic pressure at 150 m/s [kPa]": 0.5 * RHO * 150 ** 2 / 1e3, "Cd0": AF0.cd0}
SPEEDS["catapult energy [J]"] = 0.5 * m_full * SPEEDS["catapult release [m/s]"] ** 2
SPEEDS["MERLIN's stand (2.3 kg at 12 m/s) [J]"] = 0.5 * 2.3 * 12 ** 2
pd.Series(SPEEDS).round(3)

## 4. Fuel and tank for 5–30 km

For each distance, `F.size_for` finds the thinnest fuselage (in 5 mm steps) whose tank carries the fuel for the round
trip with the reserve, and `F.fuel_for` finds the smallest fuel load that still keeps 15 % at the end. The aircraft flies
the whole mission (`F.fly`) at every step of the search. **The design is sized for `DESIGN_RANGE_KM`.** A shorter
mission simply loads less fuel: a lighter aircraft, a slower catapult launch, the same airframe.

In [ ]:
rows, SIZED = {}, {}
for d in DISTANCES_KM:
    q, a, f = F.size_for(P0, UNIT, d * 1000.0, MISSION, budget=BUDGET)
    SIZED[d] = (q, a, f)
    rows[d] = {"fuselage Ø [mm]": q["fuselage_diameter"], "tank [kg]": a.tank_kg, **f.summary()}
SIZING = pd.DataFrame(rows).T.drop(columns=["distance [km]"])
display(SIZING)
P, AF, _ = SIZED[DESIGN_RANGE_KM]
if not SIZED[DESIGN_RANGE_KM][2].feasible:
    print(f"no fuselage up to 180 mm carries the fuel for {DESIGN_RANGE_KM} km: shorten DESIGN_RANGE_KM or pick a bigger engine")
FLIGHTS = {d: F.fuel_for(AF, UNIT, d * 1000.0, MISSION) for d in DISTANCES_KM}      # the design aircraft, fuel per mission
RACE = pd.DataFrame({d: f.summary() for d, f in FLIGHTS.items()}).T.drop(columns=["distance [km]"])
print(f"design: fuselage Ø {P['fuselage_diameter']:.0f} mm, tank {AF.tank_kg:.2f} kg; fuel per mission on that aircraft:")
RACE

## 5. The race against MERLIN

MERLIN's numbers come from notebook 25–27's propulsor libraries and notebook 26's design export: the best ducted fan,
tractor and pusher at the 1900 W limit. The reach time is from launch to over the fire, the moment the first reading
arrives by radio.

In [ ]:
MERLIN = None
try:
    import merlin_flight as mf, merlin_race as mr, propulsor_maps as pm
    lib, design = pm.load(), mf.load_design()
    best = {}
    for e, layout in mr.candidates(lib):
        kind = "edf" if layout == "edf" else layout
        af_m = pm.merlin_airframe(layout, 1900.0, e, design=design)
        u = pm.unit(e, 1900.0, layout=None if layout == "edf" else layout)
        t10 = mf.race(af_m, u, 10000.0)["time_to_fire_s"]
        if np.isfinite(t10) and (kind not in best or t10 < best[kind][0]):
            best[kind] = (t10, af_m, u, e["id"])
    MERLIN = {}
    for kind, (_, af_m, u, eid) in best.items():
        rr = [mf.race(af_m, u, d * 1000.0) for d in DISTANCES_KM]
        MERLIN[kind] = {"id": eid, "reach": [r["time_to_fire_s"] for r in rr], "return": [r["return_time_s"] for r in rr]}
except (FileNotFoundError, ImportError, KeyError) as exc:
    print("MERLIN comparison skipped (run notebooks 25-27 first):", exc)

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(DISTANCES_KM, RACE["to the fire [s]"], "o-", color=AGUYA_COLOR, lw=3, label="AGUYA (turbojet)")
if MERLIN:
    for kind, r in MERLIN.items():
        ax.plot(DISTANCES_KM, r["reach"], "s--", color=MERLIN_COLOR[kind], label=f"MERLIN {kind} ({r['id']}, 1900 W)")
ax.set(xlabel="distance to the fire [km]", ylabel="launch to over the fire [s]", title="reach time"); ax.grid(alpha=0.3); ax.legend(fontsize=8)
if MERLIN:
    cmp = pd.DataFrame({"AGUYA reach [s]": RACE["to the fire [s]"].values,
                        **{f"MERLIN {k} reach [s]": v["reach"] for k, v in MERLIN.items()}}, index=list(DISTANCES_KM))
    cmp["AGUYA gain vs best MERLIN"] = cmp.filter(like="MERLIN").min(axis=1) / cmp["AGUYA reach [s]"]
    display(cmp.round(1))

## 6. The mission flown

The design-range mission, step by step: catapult release, the climb at the launch speed, the acceleration to the dash
speed, the dash, the slow banked passes over the fire (the engine near idle), the dash home, and the parachute. The
mass falls as the fuel burns. The fuel flow is highest in the dash: the engine runs close to full speed there.

In [ ]:
fl = FLIGHTS[DESIGN_RANGE_KM]
tr = fl.track
fig, ax = plt.subplots(4, 1, figsize=(12, 9), sharex=True)
ax[0].plot(tr["t"], tr["V"], color=AGUYA_COLOR); ax[0].set_ylabel("airspeed [m/s]")
ax[1].plot(tr["t"], tr["h"], "k"); ax[1].set_ylabel("height [m]")
ax[2].plot(tr["t"], tr["mass"], "k"); ax[2].set_ylabel("mass [kg]")
ax[3].plot(tr["t"], tr["fuel_flow"] * 60e3, "C1"); ax[3].set_ylabel("fuel [g/min]"); ax[3].set_xlabel("time [s]")
t0 = 0.0
for name, (dur, fuel, dist) in fl.phases.items():
    for a in ax: a.axvspan(t0, t0 + dur, color=f"C{list(fl.phases).index(name)}", alpha=0.07)
    ax[0].text(t0 + dur / 2, ax[0].get_ylim()[1] * 0.92, name, ha="center", fontsize=8)
    t0 += dur
for a in ax: a.grid(alpha=0.3)
fig.tight_layout()
pd.DataFrame({k: {"time [s]": v[0], "fuel [g]": v[1] * 1000, "distance [km]": v[2] / 1000} for k, v in fl.phases.items()}).T.round(1)

## 7. The hot jet and the V-tail

The jet leaves the nozzle at about 800 K and 500 m/s, and the V-tail sits about 0.3 m behind it. Composite tails
soften above the resin's glass transition (epoxy: about 350–390 K), so the question is how hot the air is where the
tail is.

**Estimate first.** A round hot jet keeps its core temperature for about 5 nozzle diameters. Then its excess temperature
falls as `≈ 5 d / x` on the axis, and it spreads with a half-width of about `0.1 x`. The free stream at 150 m/s slows
the mixing (the velocity ratio is about 0.3), which this estimate ignores. The CFD below does not.

In [ ]:
L = ag.Aguya.layout(P)
dash_mass = AF.dry_mass_kg + 0.6 * FLIGHTS[DESIGN_RANGE_KM].fuel_kg
v_dash = FLIGHTS[DESIGN_RANGE_KM].dash_speed
rpm_dash = UNIT.for_thrust(v_dash, AF.drag(v_dash, dash_mass))[1]
JP = mj.solve(ENGINE, rpm_dash, v_dash)
d_j = P["nozzle_diameter"] / 1000
dT0 = JP.jet_temperature - 288.15

def jet_excess(x):
    x = np.asarray(x, float)
    return dT0 * np.where(x < 5 * d_j, 1.0, 5 * d_j / np.maximum(x, 1e-9))

x_le, x_te = (L["x_tail_le"] - L["x_exit"]) / 1000, (L["x_end"] - L["x_exit"]) / 1000
H = (L["R"] + L["nacelle_z"]) / 1000                                       # jet axis above the fuselage axis
gap = H * math.cos(math.radians(P["vtail_dihedral_deg"]))                  # jet axis to the V-tail planes
half = lambda x: d_j / 2 + 0.1 * x                                         # noqa: E731
print(f"dash: {v_dash:.0f} m/s at {rpm_dash / 1e3:.0f} krpm; jet {JP.jet_velocity:.0f} m/s, {JP.jet_temperature:.0f} K, "
      f"{JP.mass_flow + JP.fuel_flow:.3f} kg/s")
print(f"V-tail from {x_le:.2f} to {x_te:.2f} m behind the nozzle; jet axis {gap * 1000:.0f} mm from the tail planes; "
      f"jet half-width there {half(x_le) * 1000:.0f}-{half(x_te) * 1000:.0f} mm")
print(f"on the jet's axis: +{float(jet_excess(x_le)):.0f} K at the tail's leading edge, +{float(jet_excess(x_te)):.0f} K at its end "
      "(the tail surfaces are outside the jet's half-width when the gap exceeds it)")

### The same in CFD (Aeromant `jet_external`, optional)

The whole aircraft at the dash speed, compressible (Mach 0.44). The engine draws its air through the intake face and
blows the cycle's jet out of the nozzle face, at the cycle's flow and temperature. A tracer marks the exhaust.

What the run gives:
- the airframe's drag with the engine running, against the build-up;
- the jet's temperature and exhaust fraction along its axis.

The jet values come from the cycle at the dash point. `ag.cfd_surfaces` splits the CAD into the three STLs.

In [ ]:
CFD = None
SURF = ag.cfd_surfaces(P, ROOT / "cfd_stl")
print("faces:", SURF["faces"])

def jet_case(workdir):
    return aeromant.CFDCase("jet_external", SURF["paths"]["body"],
        dict(velocity=float(v_dash), pressure=101325.0, temperature=288.15, reference_area=AF.wing_area_m2,
             reference_length=P["span"] / 4000, center_of_rotation=[0.08, 0.0, 0.0],
             intake_mass_flow=float(JP.mass_flow), exhaust_mass_flow=float(JP.mass_flow + JP.fuel_flow),
             exhaust_temperature=float(JP.jet_temperature), iterations=300 if QUICK else 1500, residual_target=1e-5,
             cells_per_length=2.0, surface_level=3 if QUICK else 4, near_level=2 if QUICK else 3, wake_level=2,
             engine_level=3 if QUICK else 5, plume_level=2 if QUICK else 4, plume_length=4.0),
        workdir=workdir, geometry_units="mm", environment=aeromant.OpenFOAMEnvironment.detect(),
        surfaces={"intake": SURF["paths"]["intake"], "exhaust": SURF["paths"]["exhaust"]})

if RUN_CFD:
    case = jet_case(ROOT / "cfd" / "dash")
    prep = case.prepare()
    res = case.run(progress=True, processors=CPU) if prep.ok else prep
    CFD = res.metrics if res.ok else None
    if CFD:
        q = 0.5 * RHO * v_dash ** 2
        print(f"drag {CFD['drag_force_N']:.1f} N with the engine running (build-up {AF.drag(v_dash, dash_mass):.1f} N); "
              f"Cd {CFD['Cd']:.4f}, Mach {CFD['mach_number']:.2f}")
        pl = CFD.get("plume")
        if pl:
            x = np.array(pl["distance_m"])
            fig, ax = plt.subplots(1, 2, figsize=(13, 4))
            ax[0].plot(x, pl["excess_temperature_K"], "k-", label="CFD, jet axis")
            ax[0].plot(x, jet_excess(x), "k:", label="free-jet estimate")
            ax[0].axvspan(x_le, x_te, color="C0", alpha=0.15, label="V-tail")
            ax[0].set(xlabel="behind the nozzle [m]", ylabel="excess temperature [K]"); ax[0].legend()
            ax[1].semilogy(x, np.maximum(pl["exhaust_fraction"], 1e-6), "C1-"); ax[1].axvspan(x_le, x_te, color="C0", alpha=0.15)
            ax[1].set(xlabel="behind the nozzle [m]", ylabel="exhaust fraction (tracer)")
            for a in ax: a.grid(alpha=0.3)
            fig.tight_layout()
    else:
        print("jet CFD failed:", res.messages[-3:])
else:
    print("CFD skipped (VEGETA_SKIP_OPENFOAM=1): the free-jet estimate stands")

## 8. Does AGUYA sample its own exhaust?

The sensor is in the nose, 1.3 m ahead of the nozzle, so in straight flight it never sees the jet. The risk is in the
**passes**. Between crosswind passes the aircraft turns back, and it can fly through the trail it left on the previous
pass. That trail has spread for a lap's time.

The engine idles in the passes: about 50 g/min of fuel at about 75 % burner efficiency. Kerosene burned completely
gives 3.15 kg of CO₂ per kg. Its CO emission at idle is taken as 100 g per kg of fuel, a pessimistic small-turbine value.
The trail's increment is the emission per metre flown, spread over the trail's cross-section, which grows to
roughly a lap's turbulent diffusion.

In [ ]:
v_s = 1.3 * AF.stall_speed(AF.dry_mass_kg + 0.5 * FLIGHTS[DESIGN_RANGE_KM].fuel_kg, n=1 / math.cos(math.radians(30)))
ff_idle = UNIT.idle(v_s)[1]
radius = v_s ** 2 / (G * math.tan(math.radians(MISSION.sampling_bank_deg)))
lap_s = 2 * math.pi * radius / v_s + 2 * 400 / v_s                          # two 400 m passes and two half-turns
sigma = math.sqrt(2 * 1.0 * lap_s)                                         # m, eddy diffusivity ~1 m²/s at 150 m
air = math.pi * (2 * sigma) ** 2                                           # trail cross-section [m²]
ppm = lambda g_per_m3, molar: g_per_m3 / (molar / 0.0224) * 1e6 * (273.15 / 288.15)      # noqa: E731
co2 = ppm(3.15 * ff_idle * 1000 / v_s / air, 44.0)
co = ppm(0.100 * ff_idle * 1000 / v_s / air, 28.0)
pd.Series({"pass speed [m/s]": v_s, "turn radius [m]": radius, "lap [s]": lap_s, "trail diameter after a lap [m]": 4 * sigma,
           "idle fuel [g/min]": ff_idle * 60e3, "CO₂ increment in the old trail [ppm]": co2, "CO increment [ppm]": co,
           "typical CO₂ excess in a smoke column at 150 m [ppm]": 20.0, "typical CO excess [ppm]": 2.0}).round(3)

The increments are small next to a smoke column's excess (tens of ppm of CO₂, a few ppm of CO). At least, they are
small once the trail has spread for a lap. A pass flown straight back along the previous one within seconds would see
far more. Keep the passes offset by more than the turn diameter, or fly them all in one direction (a racetrack).

## 9. The wing at the dash: gust loads (Talos) and vibration modes

Pratt's gust formula, with notebook 26's numbers and AGUYA's speed:
- **Load factor**: a 7.5 m/s vertical gust at the dash speed gives `Δn = K_g ρ V U a S / (2 W)`. A small, fast aircraft
  feels gusts in proportion to its speed, so the dash takes the wing to more than 10 g.
- **Gust penetration speed**: the speed at which the same gust gives the 6 g the wing is sized for. Fly the dash below
  it in rough air.

The wing structure is a solid-equivalent glass/carbon skin over a foam core. The values are an assumption for a first
look; replace them with a laminate calculation.

The free-vibration modes (`solve_modes`, clamped at the centre section) are the input for a flutter check. The
**reduced frequency** `k = ω c / (2V)` of the first torsion mode below about 0.1 at the dash speed means the wing needs a
proper flutter analysis.

In [ ]:
COMPOSITE = talos.Material("glass/carbon skin on foam (solid-equivalent)", youngs_modulus=6000.0, poissons_ratio=0.3,
                           density=0.45e-9, yield_strength=45.0, source="assumed; replace with a laminate calculation")
W = (AF.dry_mass_kg + AF.tank_kg) * G
S_W, c_mean = AF.wing_area_m2, P["root_chord"] * (1 + P["taper"]) / 2000
A_LIFT = 2 * math.pi * AF.aspect_ratio / (AF.aspect_ratio + 2)
MU = 2 * W / S_W / (RHO * c_mean * A_LIFT * G)
K_G = 0.88 * MU / (5.3 + MU)
U_GUST, N_LIMIT = 7.5, 6.0
N_GUST = 1 + K_G * RHO * v_dash * U_GUST * A_LIFT * S_W / (2 * W)
V_PEN = (N_LIMIT - 1) * 2 * W / (K_G * RHO * U_GUST * A_LIFT * S_W)
print(f"gust load factor at {v_dash:.0f} m/s: {N_GUST:.1f} g (alleviation {K_G:.2f}); "
      f"gust penetration speed for {N_LIMIT:.0f} g: {V_PEN:.0f} m/s")

def lower_skins(step):
    info = talos.inspect_step(step, units="mm-N-MPa")
    skins = [s for s in info.surfaces if s.kind == "BSpline surface" and s.area > 5e4]
    return [min((s for s in skins if (s.centroid[1] > 0) == right), key=lambda s: s.centroid[2]).tag for right in (False, True)]

def wing_model(rev, n, name):
    yc = rev.params["fuselage_diameter"] / 2 + 5.0
    regions = [talos.SurfacesInBox("root", (-1.0, -yc - 0.1, -100.0, 400.0, yc + 0.1, 100.0)), talos.Surfaces("lift", lower_skins(rev.step))]
    return talos.StructuralModel(rev.step, "mm-N-MPa", COMPOSITE, regions, [talos.FixedSupport("root")],
                                 [talos.Pressure("lift", n * W / (S_W * 1e6))], talos.MeshSettings(element_size=6.0 if QUICK else 4.0),
                                 name=name)

def gust_at_dash(rev):  return wing_model(rev, N_GUST, "gust_at_dash")
def limit_6g(rev):      return wing_model(rev, N_LIMIT, "limit_6g")

ws = core.Workspace.create(ROOT / "workspace", name="AGUYA")
design = ws.add_design("aguya", f"{design_file}:Aguya")
wing = design.new_revision(**dict(P, part="wing"), note=f"wing, fuselage Ø {P['fuselage_diameter']:.0f} mm")
wing.generate()
FEA = {case.__name__: wing.run_fea(case.__name__, case, progress=True) for case in (limit_6g, gust_at_dash)}
display(pd.DataFrame({k: {"ok": ev.ok, "tip deflection [mm]": ev.metrics.get("max_displacement"),
                          "max von Mises [MPa]": ev.metrics.get("max_von_mises"), "safety factor": ev.metrics.get("safety_factor_yield")}
                      for k, ev in FEA.items()}).T)
MODES = None
try:
    modal = limit_6g(wing)
    mdir = ROOT / "modes"
    if modal.mesh(mdir).ok:
        r = modal.solve_modes(mdir, n_modes=6)
        if r.ok:
            f_hz = np.array(r.metrics["frequencies_hz"])
            MODES = pd.DataFrame({"frequency [Hz]": f_hz, "reduced frequency at the dash": 2 * math.pi * f_hz * c_mean / (2 * v_dash)})
            display(MODES.round(3))
        else:
            print("modal analysis failed:", r.messages[-2:])
except Exception as exc:                              # CalculiX or Gmsh missing
    print("modal analysis not run:", exc)
ws.status()

In [ ]:
ev = FEA.get("gust_at_dash")
if ev is not None and ev.ok and ev.tool_results:
    tviz.show(tviz.plot_results(ev.tool_results[-1], field="von_mises"))
else:
    print("no gust result to plot (the FEA did not run here: is CalculiX installed?)")

## 10. Export

`designs/data/aguya_design.json` holds:
- the design parameters and the airframe numbers;
- the mission table and the sizing table;
- the jet and structural checks.

In [ ]:
def clean(v):
    if isinstance(v, (np.floating, np.integer)): return v.item()
    if isinstance(v, np.ndarray): return v.tolist()
    if isinstance(v, dict): return {str(k): clean(x) for k, x in v.items()}
    if isinstance(v, (list, tuple)): return [clean(x) for x in v]
    return v

out = {"params": dict(P), "engine": ENGINE.to_dict(), "engine_source": str(JET_PATH) if JET else "catalogue (notebook 28 not run)",
       "airframe": {"dry_mass_kg": AF.dry_mass_kg, "tank_kg": AF.tank_kg, "wing_area_m2": AF.wing_area_m2,
                    "aspect_ratio": AF.aspect_ratio, "cd0": AF.cd0, "oswald": AF.oswald, "cl_max": AF.cl_max},
       "mass_budget_kg": BUDGET, "speeds": SPEEDS, "race": RACE.reset_index().to_dict("records"),
       "sizing": SIZING.reset_index().to_dict("records"), "design_range_km": DESIGN_RANGE_KM,
       "jet": {"dash_speed": v_dash, "jet_temperature_K": JP.jet_temperature, "jet_velocity": JP.jet_velocity,
               "estimate_excess_K_at_tail_le": float(jet_excess(x_le)), "cfd": CFD and {k: v for k, v in CFD.items() if k != "plume"}},
       "gust": {"load_factor": N_GUST, "penetration_speed": V_PEN},
       "fea": {k: {"ok": ev.ok, **{m: ev.metrics.get(m) for m in ("max_displacement", "max_von_mises", "safety_factor_yield")}}
               for k, ev in FEA.items()},
       "modes_hz": None if MODES is None else MODES["frequency [Hz]"].tolist(),
       "merlin_comparison": MERLIN and {k: {"id": v["id"], "reach_s": v["reach"]} for k, v in MERLIN.items()}}
path = DATA / "aguya_design.json"
path.write_text(json.dumps(clean(out), indent=1, default=str))
print("wrote", path)